# Results for model: tokyotech-llm/llama-3-swallow-70b-instruct-v0.1

In [ ]:
import polars as pl
import numpy as np
from xgboost import XGBRegressor

# Load data
train_data = pl.read_parquets(file='jane_street_data/train.parquet')
train_data = train_data.to_pandas()

# Calculate rolling 15% quantile of 'feature_00'
def calculate_quantile(group_data):
    global_quantile = group_data['feature_00'].quantile(.15)
    return global_quantile

train_data['rolling_quant_00'] = train_data.groupby('date_id')[['feature_00']].apply(calculate_quantile)
train_data = train_data.dropna()

# Train XGBoost Regressor
X = train_data.drop(['date_id', 'responder_6'], axis=1)
y = train_data['responder_6']

model = XGBRegressor(n_estimators = 500, max_depth = 6, learning_rate = 0.1)
model.fit(X, y)